# Salary Prediction: EDA, Cleaning, Modelling

**Goal:** predict `Salary` from a person's age, years of experience, education level and job title, using a **Decision Tree** and a **Random Forest**.

**How to run in Google Colab:** click the folder icon in the left sidebar, upload `salary.csv`, then run the cells from top to bottom (Runtime > Run all).
**How to run in VS Code:** keep this file in the `notebooks/` folder; the data is read from `../data/salary.csv`.

**Steps:** 1 Setup, 2 Load, 3 Quality checks, 4 Cleaning, 5 EDA, 6 Feature selection, 7 Train/test split, 8 Encoding and scaling, 9 Baseline models, 10 Tuning, 11 Final test evaluation, 12 Feature importance, 13 Example prediction, 14 Save.

## 1. Setup
Import the libraries once, at the top. `RANDOM_STATE` is a fixed number that makes every random step (splits, trees) give the **same result each time you run it**.

In [ ]:
import os, joblib                      # os: check files exist, joblib: save the model
import numpy as np                      # numbers/arrays
import pandas as pd                     # tables (DataFrames)
import matplotlib.pyplot as plt         # plotting
import seaborn as sns                   # nicer statistical plots

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer       # different preprocessing per column
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline               # chain preprocessing + model
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")        # clean plot style
RANDOM_STATE = 42                       # makes results repeatable

## 2. Load the data
`DATA_PATH` points to the CSV. The `if` line switches to the VS Code location when the Colab path doesn't exist.
`df.shape` gives (rows, columns), `df.head()` shows the first 5 rows, and `df.info()` shows column types and non-null counts.

In [ ]:
DATA_PATH = "salary.csv"                    # Colab: file uploaded to the Files panel
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../data/salary.csv"        # VS Code: notebook is in notebooks/, data is in data/

df = pd.read_csv(DATA_PATH)                 # read the CSV into a DataFrame
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.info()

## 3. Data quality checks
Before changing anything, look for problems:
- **Missing values** (blank cells)
- **Duplicate rows** (the first column `Unnamed: 0` is just a row number, so it is ignored when looking for duplicates)
- **Inconsistent category spellings**, for example `Bachelor's` vs `Bachelor's Degree`

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

# Ignore the first column (a row counter) when counting duplicates
print("\nDuplicate rows:", df.drop(columns=df.columns[0]).duplicated().sum())

# Look at every value in the small categorical columns
for col in ["Gender", "Education Level", "Country", "Race"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

print("\nUnique job titles:", df["Job Title"].nunique())

**What we found:**
- A handful of missing values (only 6 rows contain any).
- Many duplicate rows (over 1,500). Exact repeats are a problem because the same row could land in both train and test sets, which makes the test score look better than it really is.
- `Education Level` has inconsistent spellings: `Bachelor's` / `Bachelor's Degree`, `Master's` / `Master's Degree`, and `phD` / `PhD`.
- Job titles have small typos (`Juniour`, `Front end` vs `Front End`).

## 4. Cleaning
Each step prints how many rows are left, so you can see what each fix does.
Also check for impossible salaries: the code below shows rows under 1,000. The next-lowest salary in the data is 25,000, so these are almost certainly entry errors.

In [ ]:
print("Rows at start:", len(df))

df = df.drop(columns=["Unnamed: 0"])                 # 1. remove the useless row-counter column

df = df.dropna()                                     # 2. drop rows with any missing value (only 6 rows)
print("After dropping missing values:", len(df))

df["Education Level"] = df["Education Level"].replace({   # 3. make spellings consistent
    "Bachelor's": "Bachelor's Degree",
    "Master's": "Master's Degree",
    "phD": "PhD",
})

df["Job Title"] = (df["Job Title"].str.strip()             # 4. fix job-title typos
                   .str.replace("Juniour", "Junior")
                   .str.replace("Front end", "Front End"))

print("\nSuspiciously low salaries:")
print(df[df["Salary"] < 1000][["Age", "Education Level", "Job Title", "Salary"]])
df = df[df["Salary"] >= 1000]                        # 5. remove those impossible salaries
print("After removing impossible salaries:", len(df))

df = df.drop_duplicates()                            # 6. remove exact repeated rows
print("After removing duplicates:", len(df))

df = df.reset_index(drop=True)                       # 7. tidy the row numbers 0,1,2,...
print("\nFinal clean shape:", df.shape)
print(df["Education Level"].value_counts())

## 5. Exploratory Data Analysis (EDA)
EDA means looking at the data to understand it before modelling.

**5.1 Summary statistics** show count, mean, spread and min/max of each numeric column.

In [ ]:
df.describe().T

**5.2 Distributions.** Histograms show the shape of each numeric column (the line is a smoothed estimate). We are checking for skew and strange values.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Salary", "Age", "Years of Experience"]):
    sns.histplot(data=df, x=col, kde=True, ax=ax)         # bars + smooth curve
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

**5.3 Outlier check.** The IQR rule flags values far outside the middle 50% of the data. Because we already removed the impossible salaries, we expect few or none.

In [ ]:
q1, q3 = df["Salary"].quantile([0.25, 0.75])         # 25th and 75th percentiles
iqr = q3 - q1                                        # spread of the middle 50%
outliers = (df["Salary"] < q1 - 1.5 * iqr) | (df["Salary"] > q3 + 1.5 * iqr)
print("Salary outliers (IQR rule):", outliers.sum())

**5.4 Salary vs education.** Boxplots show the median (middle line), the middle 50% (the box) and the spread. Education is clearly ordered: higher education, higher salary. That is why we will encode it as an **ordered** number later.

In [ ]:
edu_order = ["High School", "Bachelor's Degree", "Master's Degree", "PhD"]   # lowest to highest

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Education Level", y="Salary", order=edu_order)
plt.title("Salary by education level")
plt.show()

**5.5 Salary vs experience.** Each dot is one person. A strong upward pattern means experience is a powerful predictor.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Years of Experience", y="Salary",
                hue="Education Level", hue_order=edu_order, alpha=0.6)
plt.title("Salary vs years of experience")
plt.show()

**5.6 Correlations.** Correlation runs from -1 to 1. Values near 1 mean two columns rise together. Note that **Age and Years of Experience are highly correlated** with each other (they carry similar information). That is fine for tree models, but would be a problem for plain linear regression.

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(df[["Age", "Years of Experience", "Salary"]].corr(),
            annot=True, fmt=".2f", cmap="Blues")      # annot=True prints the numbers
plt.title("Correlation matrix")
plt.show()

**5.7 Job titles.** There are about 190 different titles, and many appear only once or twice. The chart shows average salary for the 15 most common ones.

In [ ]:
top_titles = df["Job Title"].value_counts().head(15).index          # 15 most common titles
avg_by_title = (df[df["Job Title"].isin(top_titles)]
                .groupby("Job Title")["Salary"].mean().sort_values())

avg_by_title.plot(kind="barh", figsize=(8, 5), title="Average salary, 15 most common job titles")
plt.xlabel("Average salary")
plt.show()

rare = (df["Job Title"].value_counts() < 5).sum()
print(f"{rare} of {df['Job Title'].nunique()} job titles appear fewer than 5 times")

## 6. Feature selection: which columns should the model use?
We keep **Age, Years of Experience, Education Level, Job Title**.

We leave out **Gender, Race and Country** for two reasons:
1. **Fairness.** If the model uses gender or race, it learns and repeats pay gaps found in historical data. A salary tool that gives different answers for different genders or races is a real risk in production.
2. **No real gain.** The test below shows they don't make predictions meaningfully better. The raw averages differ between groups (table below), which may partly reflect who holds which jobs, something Job Title, Education and Experience already capture.

First, a helper that builds the preprocessing + model pipeline (explained fully in section 8):

In [ ]:
def make_pipeline(model, nominal_cols):
    # nominal_cols = unordered text columns to one-hot encode
    pre = ColumnTransformer(
        [
            # Education: ordered numbers 0,1,2,3 following edu_order
            ("edu", OrdinalEncoder(categories=[edu_order],
                                   handle_unknown="use_encoded_value",
                                   unknown_value=-1), ["Education Level"]),
            # Unordered text columns: one 0/1 column per value; values seen < 5 times are grouped
            ("nom", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=5), nominal_cols),
        ],
        remainder="passthrough",     # Age and Years of Experience pass through unchanged
    )
    return Pipeline([("pre", pre), ("model", model)])

In [ ]:
# Average salary per group (raw, not adjusted for job/experience)
for col in ["Gender", "Country", "Race"]:
    print(df.groupby(col)["Salary"].mean().round(0).sort_values(), "\n")

# Does adding these columns improve a quick Random Forest?
y_all = df["Salary"]
options = {
    "Selected features (Job Title only)": ["Job Title"],
    "Plus Gender, Country, Race": ["Job Title", "Gender", "Country", "Race"],
}
for name, nominal in options.items():
    cols = ["Age", "Years of Experience", "Education Level"] + nominal
    Xtr, Xte, ytr, yte = train_test_split(df[cols], y_all, test_size=0.2, random_state=RANDOM_STATE)
    quick = make_pipeline(RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1), nominal)
    quick.fit(Xtr, ytr)
    print(f"{name:38s} test R2 = {r2_score(yte, quick.predict(Xte)):.3f}")

## 7. Train/test split
We split **before** any encoding so the test set stays truly unseen.
- **Train (80%)**: used to learn and to tune settings.
- **Test (20%)**: touched **once**, at the very end, for an honest score.

`X` = inputs (features), `y` = the answer we want to predict.

In [ ]:
FEATURES = ["Age", "Years of Experience", "Education Level", "Job Title"]
X = df[FEATURES]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 8. Encoding and scaling
Models need numbers, not text. Two types of encoding are used:

| Column | Type | Encoding | Why |
|---|---|---|---|
| Education Level | text, **has an order** | **Ordinal / label encoding** (High School=0, Bachelor's=1, Master's=2, PhD=3) | keeps the real ranking |
| Job Title | text, **no order** | **One-hot encoding** (one 0/1 column per title) | label numbers would invent a fake order |
| Age, Years of Experience | numbers | none | already numeric |

**Normalization / scaling: not needed here.** Decision trees and Random Forests split on thresholds ("experience > 7"), so rescaling values changes nothing. Scaling matters for models like linear regression, KNN, SVM and neural networks. If you add one later, put a `StandardScaler` for the numeric columns inside the same pipeline.

**Why a pipeline?** Encoders must learn from the **training data only**; the pipeline guarantees this, including inside cross-validation (no data leakage). It also lets the API accept raw values like `"Master's Degree"` later.

Let's see what the preprocessing does:

In [ ]:
demo_pre = make_pipeline(DecisionTreeRegressor(), ["Job Title"]).named_steps["pre"]
X_demo = demo_pre.fit_transform(X_train)          # learn encodings from TRAIN data, then apply

print("Columns before:", X_train.shape[1], "-> after encoding:", X_demo.shape[1])
print("First few new column names:")
print(demo_pre.get_feature_names_out()[:5])

## 9. Baseline models with cross-validation (the validation step)
**Cross-validation (CV)** is our validation set. The training data is cut into 5 parts ("folds"). The model trains on 4 parts and is scored on the 5th, rotating 5 times. The average score tells us how well a model generalises, without touching the test set.

We score with **RMSE** (typical prediction error in salary units; lower is better).
- **Decision Tree**: one tree asking a series of yes/no questions.
- **Random Forest**: many trees on random subsets, averaged. Usually more accurate and less prone to overfitting.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)    # 5 shuffled folds

baselines = {
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

for name, model in baselines.items():
    scores = -cross_val_score(make_pipeline(model, ["Job Title"]), X_train, y_train,
                              cv=cv, scoring="neg_root_mean_squared_error")   # negative RMSE -> flip sign
    print(f"{name:14s} CV RMSE: {scores.mean():>8,.0f}  (+/- {scores.std():,.0f})")

## 10. Hyperparameter tuning
**Hyperparameters** are settings you choose before training. `GridSearchCV` tries every combination in the grid using cross-validation and keeps the best.

- `max_depth`: how deep trees may grow (too deep = memorises the data, called overfitting)
- `min_samples_leaf`: minimum rows allowed in a final leaf (bigger = simpler trees)
- `max_features` (forest only): share of columns each split may look at

The `model__` prefix tells the pipeline the setting belongs to the step named `model`. This may take a minute or two.

In [ ]:
grids = {
    "Decision Tree": (
        DecisionTreeRegressor(random_state=RANDOM_STATE),
        {"model__max_depth": [5, 10, 15, None],
         "model__min_samples_leaf": [1, 3, 5, 10]},
    ),
    "Random Forest": (
        RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
        {"model__max_depth": [10, 20, None],
         "model__min_samples_leaf": [1, 2, 4],
         "model__max_features": [0.5, 1.0]},
    ),
}

searches = {}
for name, (model, grid) in grids.items():
    search = GridSearchCV(make_pipeline(model, ["Job Title"]), grid, cv=cv,
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)              # tries all combinations, refits the best on all training data
    searches[name] = search
    print(f"{name}: best CV RMSE = {-search.best_score_:,.0f}")
    print("   best settings:", search.best_params_)

## 11. Final evaluation on the test set
Now (and only now) we score on the held-out test data.
- **R²**: share of salary variation explained (1.0 = perfect)
- **MAE**: average absolute error in salary units
- **RMSE**: like MAE but punishes big mistakes more

We also compare train vs test R². A much higher train score would signal overfitting.

In [ ]:
for name, search in searches.items():
    est = search.best_estimator_
    pred = est.predict(X_test)
    print(f"{name:14s} R2={r2_score(y_test, pred):.3f}   "
          f"MAE={mean_absolute_error(y_test, pred):>7,.0f}   "
          f"RMSE={mean_squared_error(y_test, pred) ** 0.5:>7,.0f}   "
          f"(train R2={r2_score(y_train, est.predict(X_train)):.3f})")

In [ ]:
best_name = max(searches, key=lambda n: searches[n].best_score_)       # best_score_ is negative RMSE, so highest = lowest error
best_model = searches[best_name].best_estimator_
print("Selected model (by cross-validation):", best_name)

pred = best_model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(y_test, pred, alpha=0.4)                                   # predicted vs actual
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "r--")                # perfect-prediction line
axes[0].set_xlabel("Actual salary"); axes[0].set_ylabel("Predicted salary")
axes[0].set_title("Predicted vs actual (closer to the red line = better)")

sns.histplot(y_test - pred, kde=True, ax=axes[1])                          # errors should centre on 0
axes[1].set_title("Prediction errors (actual - predicted)")
plt.tight_layout()
plt.show()

## 12. Feature importance
Which inputs does the Random Forest rely on most? (This tells us what drives the predictions. It describes the model, not cause and effect.)

In [ ]:
rf = searches["Random Forest"].best_estimator_
names = rf.named_steps["pre"].get_feature_names_out()                    # names of all encoded columns
importance = pd.Series(rf.named_steps["model"].feature_importances_, index=names)
importance.index = [n.split("__", 1)[1] for n in importance.index]       # drop the "nom__" style prefixes

importance.sort_values(ascending=False).head(10).sort_values().plot(
    kind="barh", figsize=(8, 4), title="Top 10 most important features")
plt.show()

## 13. Predict for a new person
Because preprocessing lives inside the pipeline, we can pass **raw** values. Column names must match the training columns exactly.

In [ ]:
new_person = pd.DataFrame([{
    "Age": 32,
    "Years of Experience": 5,
    "Education Level": "Bachelor's Degree",
    "Job Title": "Software Engineer",
}])
print(f"Predicted salary: {best_model.predict(new_person)[0]:,.0f}")

## 14. Save the model and summary
`joblib.dump` saves the whole fitted pipeline (preprocessing + model) into one file. `compress=3` makes it much smaller.

> **Important:** this file is only for the notebook. The model you deploy should be produced by `src/train.py` on **your own computer**, so it is saved with the same scikit-learn version your Docker image will use. A model saved with one scikit-learn version can fail or misbehave when loaded with another.

**Summary of the workflow:** load, check quality, clean, explore, choose features, split, encode inside a pipeline, compare two models with cross-validation, tune, score once on the test set, save.

In [ ]:
joblib.dump(best_model, "salary_model_notebook.pkl", compress=3)
print("Saved: salary_model_notebook.pkl")